# 05.2 — Stage 1 İzolasyon Testi
XGBoost'u tamamen bypass edip, sadece TCN+GRU'nun ham softmax çıktılarıyla Rank IC ölçer.

**Amaç:** Sinyalin Stage 1'de mi yoksa Stage 2'de mi kaybolduğunu tespit etmek.

In [ ]:
# Install dependencies
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/sergul74/Scalp2.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

from scalp2.config import load_config
from scalp2.training.walk_forward import PurgedWalkForwardCV
from scalp2.models.hybrid import HybridEncoder
from scalp2.utils.serialization import load_fold_artifacts
from scalp2.data.dataset import ScalpDataset
from torch.utils.data import DataLoader

config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'
CHECKPOINT_DIR = '/content/drive/MyDrive/scalp2/checkpoints'

print('Setup complete.')

In [ ]:
# Load dataset
df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')
with open(f'{DATA_DIR}/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

features_array = df[feature_cols].values
labels_array = df['tb_label_cls'].values

# Realized forward returns (1-bar and 10-bar)
close = df['close'].values
fwd_ret_1 = np.full(len(close), np.nan)
fwd_ret_1[:-1] = (close[1:] - close[:-1]) / close[:-1]

fwd_ret_10 = np.full(len(close), np.nan)
fwd_ret_10[:-10] = (close[10:] - close[:-10]) / close[:-10]

print(f'Dataset: {len(df)} rows, {len(feature_cols)} features')

In [ ]:
# ===== STAGE 1 İZOLASYON TESTİ =====
# XGBoost YOK — sadece TCN+GRU softmax çıktısı ile Rank IC

cv = PurgedWalkForwardCV(config.training.walk_forward)
seq_len = config.model.seq_len
n_features = len(feature_cols)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

results = []

for fold in cv.split(len(df)):
    try:
        artifacts = load_fold_artifacts(CHECKPOINT_DIR, fold.fold_idx, device=device)
    except FileNotFoundError:
        continue
    
    # Reconstruct model
    model = HybridEncoder(n_features, config.model).to(device)
    model.load_state_dict(artifacts['model_state'])
    model.eval()
    
    # Scale TEST features (same scaler as training)
    scaler = artifacts['scaler']
    test_feat = features_array[fold.test_start:fold.test_end]
    test_scaled = scaler.transform(test_feat).astype(np.float32)
    test_scaled = np.nan_to_num(test_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Create dataset and loader
    dummy_labels = np.zeros(len(test_scaled), dtype=np.int64)
    dummy_returns = np.zeros(len(test_scaled), dtype=np.float32)
    ds = ScalpDataset(test_scaled, dummy_labels, dummy_returns, seq_len)
    loader = DataLoader(ds, batch_size=512, shuffle=False, pin_memory=True)
    
    # Forward pass — get RAW LOGITS (no XGBoost!)
    all_logits = []
    all_latents = []
    with torch.no_grad():
        for batch_x, _, _ in loader:
            batch_x = batch_x.to(device)
            logits, latent = model(batch_x)
            all_logits.append(logits.cpu())
            all_latents.append(latent.cpu())
    
    logits_cat = torch.cat(all_logits, dim=0)  # (N, 3)
    probs = F.softmax(logits_cat, dim=1).numpy()  # softmax
    latents_cat = torch.cat(all_latents, dim=0).numpy()
    
    # Aligned indices (first seq_len bars are consumed by the window)
    test_offset = fold.test_start + seq_len
    n_preds = len(probs)
    
    # "Long score" = P(Long) - P(Short),  directional signal
    long_score = probs[:, 2] - probs[:, 0]
    
    # Aligned forward returns
    ret1 = fwd_ret_1[test_offset:test_offset + n_preds]
    ret10 = fwd_ret_10[test_offset:test_offset + n_preds]
    
    # Rank IC
    valid1 = ~np.isnan(ret1)
    valid10 = ~np.isnan(ret10)
    
    ic1 = spearmanr(long_score[valid1], ret1[valid1]).statistic if valid1.sum() > 30 else np.nan
    ic10 = spearmanr(long_score[valid10], ret10[valid10]).statistic if valid10.sum() > 30 else np.nan
    
    # Stage 1 accuracy (argmax of logits vs true label)
    true_labels = labels_array[test_offset:test_offset + n_preds]
    pred_labels = np.argmax(probs, axis=1)
    acc = np.mean(pred_labels == true_labels)
    
    # Class distribution of predictions
    pred_dist = np.bincount(pred_labels, minlength=3) / len(pred_labels)
    
    results.append({
        'fold': fold.fold_idx,
        'ic_1bar': ic1,
        'ic_10bar': ic10,
        'accuracy': acc,
        'pred_short_pct': pred_dist[0],
        'pred_hold_pct': pred_dist[1],
        'pred_long_pct': pred_dist[2],
        'long_score_std': np.std(long_score),
        'max_prob_mean': np.max(probs, axis=1).mean(),
    })
    
    if fold.fold_idx % 10 == 0:
        print(f'Fold {fold.fold_idx:2d}: IC(1)={ic1:+.4f}  IC(10)={ic10:+.4f}  '
              f'Acc={acc:.3f}  Pred=[S:{pred_dist[0]:.1%} H:{pred_dist[1]:.1%} L:{pred_dist[2]:.1%}]  '
              f'σ(score)={np.std(long_score):.4f}')
    
    del model
    torch.cuda.empty_cache()

rdf = pd.DataFrame(results)
print(f'\nProcessed {len(rdf)} folds')

In [ ]:
# ===== SONUÇLARI GÖSTER =====

print('=' * 70)
print('  STAGE 1 (TCN+GRU) İZOLASYON TESTİ — XGBoost YOK')
print('=' * 70)
print()
print(f'Median IC (1-bar):  {rdf["ic_1bar"].median():+.4f}')
print(f'Mean IC (1-bar):    {rdf["ic_1bar"].mean():+.4f}')
print(f'Median IC (10-bar): {rdf["ic_10bar"].median():+.4f}')
print(f'Mean IC (10-bar):   {rdf["ic_10bar"].mean():+.4f}')
print(f'IC > 0 folds (1b):  {(rdf["ic_1bar"] > 0).mean():.0%}')
print(f'IC > 0 folds (10b): {(rdf["ic_10bar"] > 0).mean():.0%}')
print()
print(f'Median Accuracy:    {rdf["accuracy"].median():.4f}')
print(f'Mean σ(long_score): {rdf["long_score_std"].mean():.4f}')
print(f'Mean max_prob:      {rdf["max_prob_mean"].mean():.4f}')
print()
print('--- Ortalama Sınıf Dağılımı (Stage 1 Tahminleri) ---')
print(f'Short: {rdf["pred_short_pct"].mean():.1%}')
print(f'Hold:  {rdf["pred_hold_pct"].mean():.1%}')
print(f'Long:  {rdf["pred_long_pct"].mean():.1%}')

# Karşılaştırma tablosu
print()
print('--- STAGE 1 vs STAGE 2 Karşılaştırma ---')
print(f'{"Metrik":<25} {"Stage1 (TCN+GRU)":>18} {"Stage2 (XGBoost)":>18}')
print('-' * 63)
# NB 5.1 sonuçları (kullanıcının paylaştığı)
s2_ic1 = 0.0133  # Median IC from NB 5.1
s2_ic10 = 0.0202
s2_acc = 0.3842
print(f'{"Median IC (1-bar)":<25} {rdf["ic_1bar"].median():>+18.4f} {s2_ic1:>+18.4f}')
print(f'{"Median IC (10-bar)":<25} {rdf["ic_10bar"].median():>+18.4f} {s2_ic10:>+18.4f}')
print(f'{"Median Accuracy":<25} {rdf["accuracy"].median():>18.4f} {s2_acc:>18.4f}')

In [ ]:
# ===== GRAFİKLER =====

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Stage 1 (TCN+GRU) İzolasyon Testi — XGBoost Bypass', fontsize=14, fontweight='bold')

# 1. Fold bazlı IC (1-bar)
ax = axes[0, 0]
colors = ['green' if x > 0 else 'red' for x in rdf['ic_1bar']]
ax.bar(rdf['fold'], rdf['ic_1bar'], color=colors, alpha=0.7)
ax.axhline(rdf['ic_1bar'].median(), color='blue', linestyle='--', label=f'Median: {rdf["ic_1bar"].median():.4f}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Fold Index')
ax.set_ylabel('Spearman IC')
ax.set_title('Stage 1 — Per-Fold IC (1-bar)')
ax.legend()

# 2. Fold bazlı IC (10-bar)
ax = axes[0, 1]
colors = ['green' if x > 0 else 'red' for x in rdf['ic_10bar']]
ax.bar(rdf['fold'], rdf['ic_10bar'], color=colors, alpha=0.7)
ax.axhline(rdf['ic_10bar'].median(), color='blue', linestyle='--', label=f'Median: {rdf["ic_10bar"].median():.4f}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Fold Index')
ax.set_ylabel('Spearman IC')
ax.set_title('Stage 1 — Per-Fold IC (10-bar)')
ax.legend()

# 3. Tahmin sınıf dağılımı
ax = axes[1, 0]
ax.bar(rdf['fold'] - 0.3, rdf['pred_short_pct'], 0.3, label='Short', color='red', alpha=0.7)
ax.bar(rdf['fold'], rdf['pred_hold_pct'], 0.3, label='Hold', color='gray', alpha=0.7)
ax.bar(rdf['fold'] + 0.3, rdf['pred_long_pct'], 0.3, label='Long', color='green', alpha=0.7)
ax.set_xlabel('Fold Index')
ax.set_ylabel('Prediction %')
ax.set_title('Stage 1 — Predicted Class Distribution')
ax.legend()
ax.set_ylim(0, 1)

# 4. Long score std (sinyal gücü)
ax = axes[1, 1]
ax.bar(rdf['fold'], rdf['long_score_std'], color='steelblue', alpha=0.7)
ax.set_xlabel('Fold Index')
ax.set_ylabel('σ(P(Long) - P(Short))')
ax.set_title('Stage 1 — Signal Strength (Long Score Std Dev)')
ax.axhline(rdf['long_score_std'].mean(), color='red', linestyle='--', label=f'Mean: {rdf["long_score_std"].mean():.4f}')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ===== YORUM =====
median_ic1 = rdf['ic_1bar'].median()
hold_pct = rdf['pred_hold_pct'].mean()
score_std = rdf['long_score_std'].mean()

print('=' * 60)
print('  OTOMATİK YORUM')
print('=' * 60)

if median_ic1 < 0.01:
    print('\n🔴 SONUÇ: Stage 1 (TCN+GRU) sinyali SIFIR.')
    print('   Sorun Stage 2 (XGBoost) değil, Stage 1\'de.')
    print('   Model fiyat hareketlerinden anlamlı pattern öğrenemiyor.')
    print()
    print('   Olası nedenler:')
    print('   1. Feature set yetersiz (MTF fix sonrası bilgi kaybı)')
    print('   2. Label ufku çok uzun (10 bar = 2.5 saat)')
    print('   3. Model kapasitesi/mimarisi uygun değil')
    print('   4. Piyasa regime\'leri çok değişken, tek model yetmiyor')
elif median_ic1 < 0.03:
    print('\n🟡 SONUÇ: Stage 1 çok zayıf ama pozitif sinyal üretiyor.')
    print('   Stage 2 (XGBoost) bunu amplify edemiyor.')
    print('   Feature/label iyileştirme gerekli.')
else:
    print('\n🟢 SONUÇ: Stage 1 anlamlı sinyal üretiyor!')
    print('   Sorun Stage 2 (XGBoost) katmanında.')
    print('   XGBoost hyperparametrelerini veya meta-feature\'ları inceleyin.')

if hold_pct > 0.7:
    print(f'\n⚠️  Model çoğunlukla Hold tahmin ediyor ({hold_pct:.0%}).')
    print('   Bu "güvenli" ama işe yaramaz — sınıf dengesizliği sorunu.')

if score_std < 0.05:
    print(f'\n⚠️  Long score std çok düşük ({score_std:.4f}).')
    print('   Softmax çıktıları neredeyse aynı — model ayrım yapamıyor.')